# Impala из Jupyter (impyla)

Нужен профиль `impala`. Impala видит те же Iceberg-таблицы, что и Spark, через общий
Hive Metastore — причём не только метаданные: `SELECT` и `INSERT` работают полноценно.
Метастор для этого собран на CDP-сборке Hive (см. README, раздел про HMS).

Подключение без SASL (`auth_mechanism='NOSASL'`).

In [ ]:
!pip install -q impyla

In [ ]:
from impala.dbapi import connect
conn = connect(host="impalad", port=21050, auth_mechanism="NOSASL")
cur = conn.cursor()
# Изменения от Spark Impala подхватывает сама через event polling (секунды).
# INVALIDATE METADATA — ручная страховка, безвредна, но обычно не нужна.
cur.execute("SHOW DATABASES")
print([r[0] for r in cur.fetchall()])

## Запрос Iceberg-таблицы, записанной Spark'ом (после make spark-demo)

In [ ]:
cur.execute("SHOW TABLES IN analytics")
print("таблицы в analytics:", [r[0] for r in cur.fetchall()])
cur.execute("SELECT * FROM analytics.sales_by_region_product ORDER BY region, product LIMIT 20")
for row in cur.fetchall():
    print(row)

## DESCRIBE FORMATTED — видно, что это Iceberg и где лежат данные

In [ ]:
cur.execute("DESCRIBE FORMATTED analytics.sales_by_region_product")
for row in cur.fetchall():
    print(row)

## Агрегация на стороне Impala

In [ ]:
cur.execute("SELECT region, round(sum(total_amount),2) FROM analytics.sales_by_region_product GROUP BY region ORDER BY region")
for row in cur.fetchall():
    print(row)

## Запись из Impala — INSERT в ту же Iceberg-таблицу

Impala не только читает: можно дописывать строки, и их сразу увидит Spark
(и наоборот — каталог-то общий).

In [ ]:
cur.execute("INSERT INTO analytics.sales_by_region_product VALUES ('TEST', 'impala-insert', 1.00)")
cur.execute("SELECT * FROM analytics.sales_by_region_product WHERE region = 'TEST'")
print(cur.fetchall())

## История снапшотов и time travel

Каждая запись (Spark или Impala — неважно) создаёт новый снапшот Iceberg.
`DESCRIBE HISTORY` показывает их, а `FOR SYSTEM_VERSION AS OF` читает таблицу на момент снапшота.

In [ ]:
cur.execute("DESCRIBE HISTORY analytics.sales_by_region_product")
snapshots = cur.fetchall()
for row in snapshots:
    print(row)

In [ ]:
# Таблица до нашего INSERT — строки TEST там ещё нет
first_snapshot = snapshots[0][1]
cur.execute(f"SELECT count(*) FROM analytics.sales_by_region_product FOR SYSTEM_VERSION AS OF {first_snapshot}")
print("строк в первом снапшоте:", cur.fetchall()[0][0])
cur.execute("SELECT count(*) FROM analytics.sales_by_region_product")
print("строк сейчас:          ", cur.fetchall()[0][0])

## Прибраться за собой

Iceberg-таблицы поддерживают `DELETE` (merge-on-read) — убираем тестовую строку.

**Нюанс:** DELETE создаёт position-delete файлы. Spark и Impala их читают, а вот
`icebergS3()` в ClickHouse 25.x — нет. Если после этого ноутбука ClickHouse откажется
читать таблицу — перезапиши её через `make spark-demo`.

In [ ]:
cur.execute("DELETE FROM analytics.sales_by_region_product WHERE region = 'TEST'")
cur.execute("SELECT count(*) FROM analytics.sales_by_region_product WHERE region = 'TEST'")
print("строк TEST осталось:", cur.fetchall()[0][0])